# Kannada-MNIST: Small CNN from Scratch

**Approach:** A compact 3-layer CNN (~300k parameters) trained from scratch to reach **97%+ accuracy** in under 5 minutes on a Colab T4 GPU.

- **Skills:** Small CNNs, training from scratch
- **No transfer learning** — the model is built and trained entirely from scratch
- **10 classes, ~65k images** (60k train + 5k test)

## Datasets
| File | Description |
|---|---|
| `train.csv` | 60,000 training samples (label + 784 pixels) |
| `test.csv` | 5,000 competition test samples (id + 784 pixels, no labels) |
| `sample_submission.csv` | Submission format (id + label) |

In [1]:
!pip install -q onnx tf2onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 15.4 MB/s eta 0:00:00


## 1. Import Libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, MaxPooling2D,
    Dropout, Flatten, Dense
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

E0000 00:00:1778008849.321819      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778008849.384312      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778008849.915075      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778008849.915121      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778008849.915123      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778008849.915126      23 computation_placer.cc:177] computation placer already registered. Please check linka

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


## 2. Load and Explore Data

In [3]:
BASE_PATH = '/kaggle/input/competitions/Kannada-MNIST/'
train_df = pd.read_csv(BASE_PATH + 'train.csv')
test_df = pd.read_csv(BASE_PATH + 'test.csv')
sample_sub = pd.read_csv(BASE_PATH + 'sample_submission.csv')

print('train_df shape:', train_df.shape)
print('test_df shape:', test_df.shape)
print('sample_sub shape:', sample_sub.shape)

train_df shape: (60000, 785)
test_df shape: (5000, 785)
sample_sub shape: (5000, 2)


In [4]:
# Peek at the data
print('\n--- train_df ---')
print(train_df.head())
print('\nLabel distribution (train):')
print(train_df['label'].value_counts().sort_index())


--- train_df ---
   label  pixel0  pixel1  pixel2  pixel3  pixel4  pixel5  pixel6  pixel7  \
0      0       0       0       0       0       0       0       0       0   
1      1       0       0       0       0       0       0       0       0   
2      2       0       0       0       0       0       0       0       0   
3      3       0       0       0       0       0       0       0       0   
4      4       0       0       0       0       0       0       0       0   

   pixel8  ...  pixel774  pixel775  pixel776  pixel777  pixel778  pixel779  \
0       0  ...         0         0         0         0         0         0   
1       0  ...         0         0         0         0         0         0   
2       0  ...         0         0         0         0         0         0   
3       0  ...         0         0         0         0         0         0   
4       0  ...         0         0         0         0         0         0   

   pixel780  pixel781  pixel782  pixel783  
0         0 

## 3. Data Preprocessing

In [5]:
# --- Training data ---
y_full = train_df['label'].values
X_full = train_df.drop('label', axis=1).values.astype('float32') / 255.0

# --- Competition test data (no labels) ---
X_test_comp = test_df.drop('id', axis=1).values.astype('float32') / 255.0

# Train / validation split (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val  : {X_val.shape}  y_val  : {y_val.shape}')
print(f'X_test_comp: {X_test_comp.shape}')

X_train: (48000, 784)  y_train: (48000,)
X_val  : (12000, 784)  y_val  : (12000,)
X_test_comp: (5000, 784)


In [6]:
# Reshape to (N, 28, 28, 1) for CNN input
X_train = X_train.reshape(-1, 28, 28, 1)
X_val   = X_val.reshape(-1, 28, 28, 1)
X_test_comp = X_test_comp.reshape(-1, 28, 28, 1)

# One-hot encode labels
num_classes = 10
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat   = to_categorical(y_val,   num_classes)

print('Shapes after reshape:')
print(f'  X_train: {X_train.shape}  y_train_cat: {y_train_cat.shape}')
print(f'  X_val  : {X_val.shape}  y_val_cat  : {y_val_cat.shape}')

Shapes after reshape:
  X_train: (48000, 28, 28, 1)  y_train_cat: (48000, 10)
  X_val  : (12000, 28, 28, 1)  y_val_cat  : (12000, 10)


## 4. Build the 3-Layer CNN (~300k params)

The architecture is intentionally kept **small and efficient**:

| Block | Layers |
|---|---|
| Conv Block 1 | Conv2D(32, 3×3) → BN → ReLU → Conv2D(32, 3×3) → BN → ReLU → MaxPool(2×2) → Dropout(0.25) |
| Conv Block 2 | Conv2D(64, 3×3) → BN → ReLU → Conv2D(64, 3×3) → BN → ReLU → MaxPool(2×2) → Dropout(0.25) |
| Conv Block 3 | Conv2D(128, 3×3) → BN → ReLU → MaxPool(2×2) → Dropout(0.25) |
| Head | Flatten → Dense(128) → BN → ReLU → Dropout(0.5) → Dense(10, softmax) |

This yields roughly **300k trainable parameters** — well within the small-CNN budget.

In [7]:
def build_model(input_shape=(28, 28, 1), num_classes=10):
    model = Sequential([
        # --- Block 1 ---
        Conv2D(32, (3, 3), padding='same', activation='relu',
               input_shape=input_shape),
        BatchNormalization(),
        Conv2D(32, (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # --- Block 2 ---
        Conv2D(64, (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(64, (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # --- Block 3 ---
        Conv2D(128, (3, 3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # --- Classification head ---
        Flatten(),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax'),
    ])
    return model

model = build_model()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1778008886.142025      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778008886.148018      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 7, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 7, 7, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 3, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 289,514 (1.10 MB)

 Trainable params: 288,618 (1.10 MB)

 Non-trainable params: 896 (3.50 KB)

## 5. Compile the Model

In [8]:
optimizer = Adam(learning_rate=1e-3)

model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

## 6. Data Augmentation & Callbacks

In [9]:
# On-the-fly data augmentation for training
datagen_train = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
)
datagen_train.fit(X_train)

# Callbacks
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3,
    verbose=1, min_lr=1e-6
)

early_stop = EarlyStopping(
    monitor='val_loss', patience=10,
    restore_best_weights=True, verbose=1
)

callbacks = [reduce_lr, early_stop]

## 7. Train the Model

With a batch size of 256 and up to 20 epochs (early stopping will kick in), the entire training should finish in **< 5 minutes on a Colab T4**.

In [10]:
batch_size = 256
epochs = 20

history = model.fit(
    datagen_train.flow(X_train, y_train_cat, batch_size=batch_size),
    steps_per_epoch=len(X_train) // batch_size,
    epochs=epochs,
    validation_data=(X_val, y_val_cat),
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1778008892.051449      75 service.cc:152] XLA service 0x7f75fc019f50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778008892.051508      75 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778008892.051515      75 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778008892.906107      75 cuda_dnn.cc:529] Loaded cuDNN version 91002


  3/187 ━━━━━━━━━━━━━━━━━━━━ 9s 50ms/step - accuracy: 0.1224 - loss: 3.3705  

I0000 00:00:1778008901.190783      75 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


187/187 ━━━━━━━━━━━━━━━━━━━━ 36s 123ms/step - accuracy: 0.6965 - loss: 0.9857 - val_accuracy: 0.1042 - val_loss: 2.9155 - learning_rate: 0.0010
Epoch 2/20
  1/187 ━━━━━━━━━━━━━━━━━━━━ 3s 21ms/step - accuracy: 0.9609 - loss: 0.1295

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


187/187 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9609 - loss: 0.1295 - val_accuracy: 0.1067 - val_loss: 2.8864 - learning_rate: 0.0010
Epoch 3/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 15s 78ms/step - accuracy: 0.9689 - loss: 0.1060 - val_accuracy: 0.7360 - val_loss: 0.7101 - learning_rate: 0.0010
Epoch 4/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9727 - loss: 0.0922 - val_accuracy: 0.7324 - val_loss: 0.7195 - learning_rate: 0.0010
Epoch 5/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 15s 81ms/step - accuracy: 0.9789 - loss: 0.0731 - val_accuracy: 0.9911 - val_loss: 0.0295 - learning_rate: 0.0010
Epoch 6/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9727 - loss: 0.1063 - val_accuracy: 0.9910 - val_loss: 0.0303 - learning_rate: 0.0010
Epoch 7/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 15s 80ms/step - accuracy: 0.9846 - loss: 0.0551 - val_accuracy: 0.9952 - val_loss: 0.0164 - learning_rate: 0.0010
Epoch 8/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9844 - loss: 0.0615 - val_a

## 8. Evaluate the Model

In [11]:
df = pd.DataFrame(history.history)
df.insert(0, 'epoch', range(1, len(df) + 1))
print(df)

    epoch  accuracy      loss  val_accuracy  val_loss  learning_rate
0       1  0.856526  0.453627      0.104250  2.915484        0.00100
1       2  0.960938  0.129500      0.106667  2.886380        0.00100
2       3  0.972415  0.092367      0.736000  0.710060        0.00100
3       4  0.972656  0.092242      0.732417  0.719501        0.00100
4       5  0.980584  0.067214      0.991083  0.029512        0.00100
5       6  0.972656  0.106286      0.991000  0.030311        0.00100
6       7  0.985297  0.050858      0.995250  0.016437        0.00100
7       8  0.984375  0.061481      0.995083  0.016413        0.00100
8       9  0.986449  0.044706      0.992833  0.023123        0.00100
9      10  0.972656  0.041320      0.993500  0.020889        0.00100
10     11  0.988690  0.037542      0.995250  0.014245        0.00050
11     12  0.984375  0.033066      0.995250  0.014179        0.00050
12     13  0.989716  0.034314      0.994833  0.015711        0.00050
13     14  0.984375  0.050666     

In [12]:
# --- Validation set accuracy ---
val_loss, val_acc = model.evaluate(X_val, y_val_cat, verbose=0)
print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_acc:.4f}  ({val_acc*100:.2f}%)')

Validation Loss: 0.0110
Validation Accuracy: 0.9965  (99.65%)


In [13]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import numpy as np

y_val_pred = model.predict(X_val, verbose=0)
y_val_pred_classes = np.argmax(y_val_pred, axis=1)

# Confusion Matrix
cm = confusion_matrix(y_val, y_val_pred_classes)

# Convert to DataFrame for nicer printing
cm_df = pd.DataFrame(cm,
                     index=[f'True_{i}' for i in range(10)],
                     columns=[f'Pred_{i}' for i in range(10)])

print("\nConfusion Matrix (Validation Set):")
print(cm_df)

# Classification Report
print("\nClassification Report (Validation):")
report = classification_report(y_val, y_val_pred_classes, output_dict=True)

# Convert to DataFrame for table format
report_df = pd.DataFrame(report).transpose()
print(report_df.round(4))


Confusion Matrix (Validation Set):
        Pred_0  Pred_1  Pred_2  Pred_3  Pred_4  Pred_5  Pred_6  Pred_7  \
True_0    1190       7       1       1       1       0       0       0   
True_1       3    1196       0       1       0       0       0       0   
True_2       0       0    1200       0       0       0       0       0   
True_3       0       0       0    1197       0       1       0       2   
True_4       0       0       0       1    1198       0       0       0   
True_5       0       0       0       0       0    1200       0       0   
True_6       0       0       0       0       0       0    1194       3   
True_7       0       0       0       4       0       0       4    1191   
True_8       1       0       0       0       0       0       0       0   
True_9       0       0       0       0       0       0       7       0   

        Pred_8  Pred_9  
True_0       0       0  
True_1       0       0  
True_2       0       0  
True_3       0       0  
True_4       0       1  

## 9. Generate Competition Submission

In [14]:
# Predict on competition test set
test_pred = model.predict(X_test_comp, verbose=0)
test_pred_classes = np.argmax(test_pred, axis=1)

submission = pd.DataFrame({
    'id': test_df['id'],
    'label': test_pred_classes
})

submission.to_csv('submission.csv', index=False)
print('Submission saved!')
print(submission.head(10))

Submission saved!
   id  label
0   0      3
1   1      0
2   2      2
3   3      6
4   4      7
5   5      7
6   6      1
7   7      9
8   8      3
9   9      4


## 10. Conclusion

A small 3-layer CNN with ~300k parameters achieves **97%+ validation accuracy** on the Kannada-MNIST dataset, trained from scratch in under 5 minutes on a Colab T4 GPU. No transfer learning was used.

Key design decisions:
- **3 convolutional blocks** with increasing filter counts (32 → 64 → 128)
- **Batch Normalization** after each Conv2D for stable, fast training
- **Dropout** (0.25 in conv blocks, 0.5 in dense head) for regularization
- **On-the-fly data augmentation** (rotation, shift, shear, zoom) to improve generalization
- **ReduceLROnPlateau + EarlyStopping** callbacks for efficient training
- **Adam optimizer** with learning rate 1e-3

## 11. Export Model and Config for Web App

In [15]:
import tf2onnx
import onnx
import tensorflow as tf

# In Keras 3, converting directly from Keras model sometimes fails with KeyError.
# A robust workaround is to convert from a tf.function trace instead:
@tf.function
def serve(x):
    return model(x)

spec = (tf.TensorSpec((None, 28, 28, 1), tf.float32, name="input"),)
output_path = "kannada_cnn.onnx"
model_proto, _ = tf2onnx.convert.from_function(serve, input_signature=spec, opset=13, output_path=output_path)
print("ONNX model saved!")


I0000 00:00:1778009075.075371      23 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1778009075.075581      23 single_machine.cc:374] Starting new session
I0000 00:00:1778009075.085783      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778009075.087241      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1778009075.217223      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778009075.218739      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 M

ONNX model saved!


In [16]:
# ── GENERATE .TS FILES FOR LOVABLE APP ─────────────────────────────────────

model_config_ts = f"""export const MODEL_CONFIG = {{
  modelName: "KannadaCNN",
  version: "1.0.0",
  inputShape: [1, 28, 28, 1],
  inputSize: 28,
  numClasses: 10,
  classLabels: ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"],
  bestValAccuracy: {val_acc * 100:.3f},
  modelPath: "/kannada_cnn.onnx" // Make sure to place the model in the public folder
}};
"""

preprocessing_ts = """export function preprocessCanvas(imageData: ImageData): Float32Array {
  const { data, width, height } = imageData;
  const tensor = new Float32Array(width * height);

  // Convert RGBA to Grayscale and normalize to [0, 1]
  for (let i = 0; i < data.length; i += 4) {
    const r = data[i];
    const g = data[i + 1];
    const b = data[i + 2];
    const a = data[i + 3];

    // Grayscale conversion based on luminance
    const gray = (r * 0.299 + g * 0.587 + b * 0.114);

    // Since images in Kannada-MNIST are typically dark strokes on light background
    // If your canvas is white-on-black, no inversion is needed.
    // Assuming user draws black on white canvas:
    let pixelValue = 255 - gray; 

    // Normalize to [0, 1] as expected by the model
    tensor[i / 4] = pixelValue / 255.0;
  }

  return tensor;
}
"""

model_loader_ts = """import * as ort from 'onnxruntime-web';
import { MODEL_CONFIG } from './model-config';
import { preprocessCanvas } from './preprocessing';

export class ModelLoader {
  private session: ort.InferenceSession | null = null;

  async loadModel() {
    try {
      this.session = await ort.InferenceSession.create(MODEL_CONFIG.modelPath, {
        executionProviders: ['wasm']
      });
      console.log("ONNX Model loaded successfully.");
    } catch (e) {
      console.error("Failed to load ONNX model", e);
    }
  }

  async predict(imageData: ImageData): Promise<number> {
    if (!this.session) {
      throw new Error("Model is not loaded yet.");
    }

    const preprocessedData = preprocessCanvas(imageData);
    const tensor = new ort.Tensor('float32', preprocessedData, MODEL_CONFIG.inputShape);

    const feeds: Record<string, ort.Tensor> = {};
    feeds[this.session.inputNames[0]] = tensor;

    const results = await this.session.run(feeds);
    const outputTensor = results[this.session.outputNames[0]];
    const outputData = outputTensor.data as Float32Array;

    // Argmax
    let maxVal = -Infinity;
    let maxIdx = -1;
    for (let i = 0; i < outputData.length; i++) {
      if (outputData[i] > maxVal) {
        maxVal = outputData[i];
        maxIdx = i;
      }
    }

    return maxIdx;
  }
}
"""

with open("model-config.ts", "w") as f: f.write(model_config_ts)
with open("preprocessing.ts", "w") as f: f.write(preprocessing_ts)
with open("model-loader.ts", "w") as f: f.write(model_loader_ts)

print("TS files (model-config.ts, preprocessing.ts, model-loader.ts) generated successfully!")

TS files (model-config.ts, preprocessing.ts, model-loader.ts) generated successfully!
